In [1]:
%load_ext cudf.pandas

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [3]:
%%RecordEvent
# %load_ext cudf.pandas
import sys, os
tpch_parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, tpch_parent)
import pandas as pd
#from tpch import load_customer, load_orders, q13
DATA_ROOT = "/home/colinc/code/tpch/data/tpch_15"#"/home/jupyter/tpch-dbgen/data"
STORAGE_OPTS = {}



In [4]:

### cell 0 ###

def load_customer(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/customer.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df

customer = load_customer(DATA_ROOT, **STORAGE_OPTS)


In [5]:

### cell 1 ###

def load_orders(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/orders.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    df["O_ORDERDATE"] = pd.to_datetime(df.O_ORDERDATE, format="%Y-%m-%d")
    return df
orders = load_orders(DATA_ROOT, **STORAGE_OPTS)

In [6]:
%%time
### cell 2 ###

# 1) Count orders per customer (exclude special requests)
cust_order_counts = (
    orders
    .loc[
        ~orders["O_COMMENT"].str.contains(r"special[\S|\s]*requests"),
        "O_CUSTKEY"
    ]
    .value_counts()
)

# 2) Build distribution of counts for customers with ≥1 order
dist = (
    cust_order_counts
    .value_counts()
    .rename("CUSTDIST")
    .reset_index()
)
dist.columns = ["C_COUNT", "CUSTDIST"]

# compute how many customers had zero orders
zeros = len(customer) - len(cust_order_counts)
if zeros:
    # use concat instead of append (unsupported in cudf.DataFrame)
    zero_df = pd.DataFrame({"C_COUNT": [0], "CUSTDIST": [zeros]})
    dist = pd.concat([dist, zero_df], ignore_index=True)

# 3) Sort and assign to total
total = dist.sort_values(["CUSTDIST", "C_COUNT"], ascending=[False, False])

CPU times: user 248 ms, sys: 499 ms, total: 747 ms
Wall time: 740 ms


In [7]:
# %%time
# #original
# customer_filtered = customer.loc[:, ["C_CUSTKEY"]]
# orders_filtered = orders[
#     ~orders["O_COMMENT"].str.contains(r"special[\S|\s]*requests")
# ]
# orders_filtered = orders_filtered.loc[:, ["O_ORDERKEY", "O_CUSTKEY"]]
# c_o_merged = customer_filtered.merge(
#     orders_filtered, left_on="C_CUSTKEY", right_on="O_CUSTKEY", how="left"
# )
# c_o_merged = c_o_merged.loc[:, ["C_CUSTKEY", "O_ORDERKEY"]]
# count_df = c_o_merged.groupby(["C_CUSTKEY"], as_index=False, sort=False).agg(
#     C_COUNT=pd.NamedAgg(column="O_ORDERKEY", aggfunc="count")
# )
# total = count_df.groupby(["C_COUNT"], as_index=False, sort=False).size()
# total.columns = ["C_COUNT", "CUSTDIST"]
# total = total.sort_values(by=["CUSTDIST", "C_COUNT"], ascending=[False, False])

CPU times: user 1.07 s, sys: 1.18 s, total: 2.25 s
Wall time: 2.24 s


In [10]:
total.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 46 entries, 2 to 40
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   C_COUNT   46 non-null     int64
 1   CUSTDIST  46 non-null     int64
dtypes: int64(2)
memory usage: 1.1 KB


In [7]:
### cell 3 ###

total.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 46 entries, 45 to 43
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   C_COUNT   46 non-null     int64
 1   CUSTDIST  46 non-null     int64
dtypes: int64(2)
memory usage: 1.1 KB


In [11]:
total

,C_COUNT,CUSTDIST
2,0,500021
15,10,66157
13,9,65243
18,11,62072
11,8,58350
12,12,55821
9,13,49811
6,7,46847
3,19,46641
8,18,46259


In [8]:
total

,C_COUNT,CUSTDIST
2,0,500021
15,10,66157
13,9,65243
18,11,62072
11,8,58350
12,12,55821
9,13,49811
6,7,46847
3,19,46641
8,18,46259
